In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/Users/folusoogunlana/.cache/huggingface/hub


In [5]:
MODEL = "models/Qwen2.5-0.5B-Instruct"
device = "mps" if torch.backends.mps.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(device).eval()

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 138.01it/s]


In [7]:
msgs = [{"role": "user", "content": "Say hello in one short sentence"}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

inputs = tok(text, return_tensors="pt").to(device)

out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
print(tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))

Hello! How can I assist you today?


In [10]:
def acts(text):
    inputs = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        hs = model(**inputs, output_hidden_states=True).hidden_states
        print(hs[0].shape)
    return torch.stack([h[0, -1] for h in hs])

a = acts("I feel wonderful today!")
b = acts("I feel miserable today!")
print(torch.nn.functional.cosine_similarity(a, b, dim=-1))

torch.Size([1, 5, 896])
torch.Size([1, 5, 896])
tensor([1.0000, 0.9943, 0.9936, 0.9910, 0.9865, 0.9732, 0.9696, 0.9665, 0.9548,
        0.9572, 0.9573, 0.9425, 0.9420, 0.9474, 0.9392, 0.9395, 0.9486, 0.9550,
        0.9599, 0.9583, 0.9558, 0.9569, 0.9714, 0.9690, 0.9589],
       device='mps:0')
